In [1]:
# --- MÓDULO DE INGESTÃO: MERCADO DE TRABALHO E VÍNCULOS FORMAIS ---

import pandas as pd

# Defino o ponto de entrada para os dados de mercado de trabalho formal.
# Esta variável mensura a geração de empregos (vínculos totais), permitindo 
# avaliar o impacto social e a absorção de mão de obra gerada pela atividade minerária.
caminho_arquivo = r'C:\Users\fabio\TCC\7_Vinculos_total.csv'

try:
    # Realizo a ingestão do dataset de estoque de empregos formais.
    # encoding='latin-1': Padrão necessário para decodificar bases do Ministério do Trabalho.
    # skipfooter=19: Remoção cirúrgica do bloco de notas metodológicas no rodapé do arquivo.
    # engine='python': Parâmetro obrigatório da biblioteca Pandas ao utilizar skipfooter.
    df_emprego = pd.read_csv(
        caminho_arquivo,
        sep=';',
        skipfooter=19,
        encoding='latin-1',
        engine='python' 
    )
    
    # DIAGNÓSTICO ESTRUTURAL INICIAL:
    # Confirmo se a carga superou as notas de rodapé e se os caracteres (acentos) 
    # foram lidos corretamente com o encoding latin-1.
    print("--- Visualização do Dataset de Vínculos Empregatícios ---")
    print(df_emprego.head())
    
    print("\n--- Inventário de Séries Temporais (Mercado de Trabalho) ---")
    print(df_emprego.columns.tolist())

except Exception as e:
    print(f"ERRO DE INGESTÃO: Falha crítica ao carregar a matriz de Empregos: {e}")

--- Visualização do Dataset de Vínculos Empregatícios ---
  Estado                 Cidade   2021   2020   2019   2018   2017   2016  \
0     RO  ALTA FLORESTA D OESTE   2631   2715   2811   2953   2900   3088   
1     RO              ARIQUEMES  18378  17534  17932  18049  18028  17977   
2     RO                 CABIXI    652    675    706    732    730    693   
3     RO                 CACOAL  18091  17526  17910  17846  17364  16832   
4     RO             CEREJEIRAS   2564   2372   2230   2351   2349   2207   

    2015   2014  ...   2010   2009   2008   2007   2006   2005   2004  2003  \
0   2917   2968  ...   2481   2224   2036   2024   1950   1715   1610  1629   
1  18761  19392  ...  16061  15379  14336  12902  11912  11131  10291  9391   
2    684    669  ...    519    490    409    423    401    397    367   370   
3  17610  17729  ...  14107  13039  11976  11547  10687  10200   9381  9119   
4   2458   2512  ...   2343   1739   1596   1520   1458   1430   1335  1346   

   2

In [2]:
# --- ETAPA 2: HIGIENIZAÇÃO E TRANSPOSIÇÃO DO PAINEL DE EMPREGOS ---

# Realizo a remoção de agregadores horizontais e a conversão do dataset
# para o formato 'Long'. Esta estrutura permite acompanhar a evolução do 
# estoque de empregos formais (vínculos) ano a ano no modelo econométrico.

print("Iniciando a higienização da matriz de mercado de trabalho...")

# 1. TRATAMENTO DE TOTALIZADORES (CROSS-SECTIONAL SUMS)
# Removo a coluna 'Total' nativa da RAIS para evitar dupla contagem 
# e distorções na agregação do painel de dados.
try:
    df_emprego_limpo = df_emprego.drop(columns=['Total'])
    print("Sucesso: Agregador horizontal 'Total' removido com segurança.")
except KeyError:
    print("Aviso: Coluna 'Total' ausente; prosseguindo com o dataset atual.")
    df_emprego_limpo = df_emprego.copy()

# 2. REESTRUTURAÇÃO PARA SÉRIE HISTÓRICA (MELT)
# Transponho a matriz para alinhar a métrica de empregos formais
# à estrutura longitudinal exigida pelo Stata/R.

# 2a. Definição das Chaves Geográficas (Atenção à divergência com o IBGE)
dimensoes_id = ['Estado', 'Cidade']

# 2b. Execução do Reshaping (Wide to Long)
df_emprego_final = df_emprego_limpo.melt(
    id_vars=dimensoes_id,
    var_name='Ano',
    value_name='Vinculos_totais'
)

# 3. AUDITORIA DA SÉRIE DE EMPREGOS
# Validação dos registros iniciais e finais para garantir a consistência
# da transposição temporal.
print("\n--- Diagnóstico: Painel de Vínculos Formais (Long Format) ---")
print("Cabeçalho (Início da Série Histórica):")
print(df_emprego_final.head())

print("\nRodapé (Período Mais Recente):")
print(df_emprego_final.tail())

Iniciando a higienização da matriz de mercado de trabalho...
Sucesso: Agregador horizontal 'Total' removido com segurança.

--- Diagnóstico: Painel de Vínculos Formais (Long Format) ---
Cabeçalho (Início da Série Histórica):
  Estado                 Cidade   Ano  Vinculos_totais
0     RO  ALTA FLORESTA D OESTE  2021             2631
1     RO              ARIQUEMES  2021            18378
2     RO                 CABIXI  2021              652
3     RO                 CACOAL  2021            18091
4     RO             CEREJEIRAS  2021             2564

Rodapé (Período Mais Recente):
       Estado          Cidade   Ano  Vinculos_totais
111415     GO      VIANOPOLIS  2002             1206
111416     GO  VICENTINOPOLIS  2002              718
111417     GO        VILA BOA  2002              266
111418     GO   VILA PROPICIO  2002              245
111419     DF        BRASILIA  2002           813591


In [3]:
# --- MÓDULO 2.5: INTEGRAÇÃO RETROATIVA DE SÉRIE HISTÓRICA (1996-2001) ---

import os
import pandas as pd

# 1. CONFIGURAÇÃO DE DIRETÓRIO LEGADO
# Estabeleço o caminho para os microdados antigos da RAIS. A recuperação deste 
# período é estratégica para expandir a janela de observação pré-tratamento do modelo.
pasta_antigos = r'C:\Users\fabio\TCC\7_Vinculos_totais_1996_2001'
lista_dfs_antigos = []

print(f"--- Iniciando Ingestão de Dados Legados (Backcasting): {pasta_antigos} ---")

# 2. PIPELINE DE EXTRAÇÃO E LIMPEZA DE ARQUIVOS HISTÓRICOS
for ano in range(1996, 2002):
    nome_arquivo = f'{ano}.csv'
    caminho_completo = os.path.join(pasta_antigos, nome_arquivo)
    
    try:
        # Ingestão com salto de cabeçalhos institucionais
        df_temp = pd.read_csv(caminho_completo, sep=';', encoding='latin-1', skiprows=1)
        
        # Referenciamento Dinâmico: 
        # Utilizo índices de coluna em vez de strings literais para mitigar erros 
        # de encoding ('Município' vs 'Municipio') comuns em bases antigas.
        col_cidade_suja = df_temp.columns[0] 
        col_valor = df_temp.columns[1]       
        
        # --- NORMALIZAÇÃO E PARSING DE STRINGS ---
        
        # A. Separação de Chaves Geográficas (String Splitting)
        # O formato legado aglutina 'UF-Município' (ex: MG-ARAGUARI). 
        # Realizo o split para garantir compatibilidade com o schema da base recente.
        dados_separados = df_temp[col_cidade_suja].str.split('-', n=1, expand=True)
        
        df_temp['Estado'] = dados_separados[0]  # Extração da UF
        df_temp['Cidade'] = dados_separados[1]  # Extração do Município
        
        # B. Padronização de Variáveis e Injeção Temporal
        df_temp = df_temp.rename(columns={col_valor: 'Vinculos_totais'})
        df_temp['Ano'] = ano 
        
        # C. Seleção de Schema (Feature Selection)
        # Isolo estritamente os atributos necessários, respeitando a ordem do painel mestre.
        df_limpo_ano = df_temp[['Estado', 'Cidade', 'Ano', 'Vinculos_totais']]
        
        lista_dfs_antigos.append(df_limpo_ano)
        print(f"   -> Sucesso: Matriz do ano {ano} normalizada.")
        
    except Exception as e:
        print(f"   -> [ERRO CRÍTICO] Falha na leitura do lote {ano}: {e}")

# 3. CONSOLIDAÇÃO MESTRA DO PAINEL DE EMPREGOS (1996-2021)

if lista_dfs_antigos:
    # Agregação do período legado
    df_antigos_consolidado = pd.concat(lista_dfs_antigos, ignore_index=True)
    
    # União Vertical (Stacking) com a série recente (2002-2021)
    df_emprego_final = pd.concat([df_emprego_final, df_antigos_consolidado], ignore_index=True)
    
    # Ordenação de Painel (Panel Sorting)
    # Fundamental para preparar a estrutura longitudinal exigida por softwares 
    # econométricos (alinhamento Unidade-Tempo).
    df_emprego_final = df_emprego_final.sort_values(by=['Cidade', 'Ano'])
    
    print("\n--- INTEGRAÇÃO DE SÉRIES CONCLUÍDA ---")
    print(f"Dimensão Final do Painel de Mercado de Trabalho: {len(df_emprego_final)} observações")
    print("\nAuditoria Inicial (Período Base - Expectativa: 1996):")
    print(df_emprego_final.head())
    print("\nAuditoria Final (Período Recente - Expectativa: 2021):")
    print(df_emprego_final.tail())

else:
    print("\n[ALERTA]: Nenhum lote histórico foi processado. Verifique o repositório.")

--- Iniciando Ingestão de Dados Legados (Backcasting): C:\Users\fabio\TCC\7_Vinculos_totais_1996_2001 ---
   -> Sucesso: Matriz do ano 1996 normalizada.
   -> Sucesso: Matriz do ano 1997 normalizada.
   -> Sucesso: Matriz do ano 1998 normalizada.
   -> Sucesso: Matriz do ano 1999 normalizada.
   -> Sucesso: Matriz do ano 2000 normalizada.
   -> Sucesso: Matriz do ano 2001 normalizada.

--- INTEGRAÇÃO DE SÉRIES CONCLUÍDA ---
Dimensão Final do Painel de Mercado de Trabalho: 116343 observações

Auditoria Inicial (Período Base - Expectativa: 1996):
       Estado           Cidade   Ano  Vinculos_totais
110773    RS    PINTO BANDEIRA  2002                0
105202    RS    PINTO BANDEIRA  2003                0
99631     RS    PINTO BANDEIRA  2004                0
94060     RS    PINTO BANDEIRA  2005                0
88489     RS    PINTO BANDEIRA  2006                0

Auditoria Final (Período Recente - Expectativa: 2021):
      Estado  Cidade   Ano  Vinculos_totais
26888     SC  ZORTEA  201

In [4]:
# --- ETAPA 3: SERIALIZAÇÃO E PERSISTÊNCIA DO PAINEL DE EMPREGOS (OUTPUT) ---

# Concluo o módulo de Mercado de Trabalho com a exportação da série histórica 
# consolidada (1996-2021). Este painel longo é o insumo principal para medir 
# a capacidade de absorção de mão de obra formal gerada pelo choque econômico.

# 1. DEFINIÇÃO DO DIRETÓRIO DE SAÍDA
# Padronizo a nomenclatura do dataset final para garantir a interoperabilidade 
# e leitura sequencial no script de integração mestre (Master Merge).
caminho_saida_csv = r'C:\Users\fabio\TCC\VARIAVEIS\7_emprego_total_FINAL.csv'

# 2. EXPORTAÇÃO E INTEGRIDADE ESTRUTURAL
# O parâmetro index=False é estritamente necessário para suprimir os índices 
# nativos da memória do Pandas, evitando a geração de colunas residuais 
# ('Unnamed: 0') durante a leitura no Stata, R ou Python.
df_emprego_final.to_csv(
    caminho_saida_csv,
    index=False
)

print(f"--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Dataset de Vínculos Formais (RAIS 1996-2021) salvo com sucesso em:")
print(caminho_saida_csv)

--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---
Dataset de Vínculos Formais (RAIS 1996-2021) salvo com sucesso em:
C:\Users\fabio\TCC\VARIAVEIS\7_emprego_total_FINAL.csv
